In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.modules.loss as L
from torch.nn.utils.rnn import pad_sequence
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from skorch import NeuralNet
from skorch.scoring import loss_scoring
import sys
from collections import OrderedDict
import pickle
import inspect

from preprocessing import preprocess_dataset
from models import LSTM2LSTM, Conv2LSTM

/home/jim/notebook_env/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


### Constants

In [2]:
MAX_WORDS = 15
MAX_SAMPLES = 2000

# TRAINING ROUTINE
This function takes in a PyTorch model encapsulated 
within a Skorch wrapper, a dictionary of possible 
hyperparameters, and some data. The algorithm for 
tuning and training a model used here is as follows:
1. Instantiate the grid search wrapper.
2. Split training and testing data.
3. Fit the estimator (hyperparameter tuning and training is done here).
4. Compute the training/testing loss.
5. Return the fully configured model along with its testing data.

In [3]:
def get_model(model, params, X, y, y_dict):

    # Define estimator for grid search
    estimator = GridSearchCV(model, params, cv=3, verbose=2)

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

    y_train = F.one_hot(y_train, num_classes=len(y_dict)).to(torch.float)
    y_test = F.one_hot(y_test, num_classes=len(y_dict)).to(torch.float)

    # Fit the estimator
    estimator.fit(X_train, y_train)

    # Gather training and testing predictions
    y_train_pred = torch.tensor(estimator.predict_proba(X_train), dtype=torch.float)
    y_test_pred = torch.tensor(estimator.predict_proba(X_test), dtype=torch.float)
    
    # Compute loss
    train_loss = F.cross_entropy(y_train_pred, y_train)
    test_loss = F.cross_entropy(y_test_pred, y_test)

    # A fully tuned and trained model, along with testing data
    return estimator, train_loss, test_loss

### Fetching data, checking GPU availability

In [4]:
if torch.cuda.is_available(): device = "cuda"
else: device = "cpu"

print("Device: ", device)

Device:  cuda


In [5]:
# Fetch data
eng_file = open("translation_corpus/Tatoeba.en-fr.en", "r", encoding="utf-8")
fre_file = open("translation_corpus/Tatoeba.en-fr.fr", "r", encoding="utf-8")

X_raw = eng_file.readlines()[:MAX_SAMPLES]
y_raw = fre_file.readlines()[:MAX_SAMPLES]

In [6]:
X, y, X_dict, y_dict = preprocess_dataset(X_raw, y_raw, max_words=MAX_WORDS)

print("X Vocab: ", len(X_dict))
print("y Vocab: ", len(y_dict))

/home/jim/notebook_env/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


X Vocab:  1876
y Vocab:  2216


# SKORCH WRAPPER
This class defines the skorch wrapper used for both models. 
It is fairly simple, the only extensions to the base class 
are a "score" method for directing the grid search, and an
extended "predict" method for returning class predictions
from probability distribution output, since the model is not
intrinsically capable of this.

In [7]:
class TrnsltNet(NeuralNet):

    def score(self, X, y):

        # Negative loss for the grid search
        return loss_scoring(self, X, y) * -1

    def predict(self, X):

        # Get probabilities
        predictions = self.predict_proba(X)

        # Get class predictions from probabilities
        predictions = np.argmax(predictions, axis=2)

        # Return class predictions
        return predictions

# THE PURE LSTM MODEL
The following cells instantiate, configure, and train a seq2seq model 
whose encoder and decoder are both LSTM-based.

In [8]:
torch.cuda.empty_cache()

lstm2lstm = TrnsltNet(module=LSTM2LSTM, 
                      module__input_space=len(X_dict),
                      module__output_space=len(y_dict),
                      module__hidden_space=len(X_dict),
                      module__layers=1,
                      module__max_words=MAX_WORDS,
                      module__device=device,
                      device=device,
                      max_epochs=10, 
                      batch_size=10,
                      train_split=None,
                      criterion=nn.CrossEntropyLoss(), 
                      optimizer=torch.optim.Adam, 
                      lr=0.01)

In [9]:
params = {
    "module__layers" : [1, 2],
    "module__activation" : [nn.ReLU(), nn.LeakyReLU()],
}

# Tune hyperparameters and train model
lstm2lstm_estimator, lstm2lstm_train_loss, lstm2lstm_test_loss = get_model(lstm2lstm, params, X, y, y_dict)

# Save hyperparameters for display
lstm2lstm_params = lstm2lstm_estimator.best_params_

# Save model
pickle.dump(lstm2lstm_estimator, open("saved_models/lstm2lstm_estimator.pkl", "wb"))

# Delete estimator to save space
del lstm2lstm_estimator

Fitting 3 folds for each of 4 candidates, totalling 12 fits
  epoch    train_loss     dur
-------  ------------  ------
      1        0.0180  7.2919
      2        0.0183  7.1513
      3        0.0183  7.2542
      4        0.0183  7.1565
      5        0.0183  7.1580
      6        0.0183  7.1533
      7        0.0183  7.1646
      8        0.0183  7.1631
      9        0.0183  7.1662
     10        0.0183  7.1615
[CV] END ........module__activation=ReLU(), module__layers=1; total time= 1.2min
  epoch    train_loss     dur
-------  ------------  ------
      1        0.0177  7.1174
      2        0.0174  7.1741
      3        0.0174  7.1764
      4        0.0174  7.1692
      5        0.0174  7.1694
      6        0.0174  7.1788
      7        0.0174  7.1705
      8        0.0174  7.1775
      9        0.0174  7.1775
     10        0.0174  7.1808
[CV] END ........module__activation=ReLU(), module__layers=1; total time= 1.2min
  epoch    train_loss     dur
-------  ------------  -----

In [10]:
print("==TRAIN LOSS==")
print(lstm2lstm_train_loss)
print("==TEST LOSS==")
print(lstm2lstm_test_loss)
print("==HYPERPARAMETERS==")
print(lstm2lstm_params)

==TRAIN LOSS==
tensor(0.0183)
==TEST LOSS==
tensor(0.0183)
==HYPERPARAMETERS==
{'module__activation': LeakyReLU(negative_slope=0.01), 'module__layers': 1}


# THE HYBRID CONVOLUTIONAL-LSTM MODEL
The following cells instantiate, configure, and train a seq2seq model 
whose encoder consists of 1-D convolutional and pooling layers, but 
whose decoder is LSTM-based.

In [11]:
torch.cuda.empty_cache()

In [12]:
conv2lstm = TrnsltNet(module=Conv2LSTM,
                                module__input_space=len(X_dict),
                                module__output_space=len(y_dict),
                                module__hidden_space=len(X_dict),
                                module__conv_kernel_size=1,
                                module__pool_kernel_size=1,
                                module__device=device,
                                device=device,
                                max_epochs=10, 
                                batch_size=10,
                                train_split=None,
                                criterion=nn.CrossEntropyLoss(), 
                                optimizer=torch.optim.Adam, 
                                lr=0.01)

In [13]:
params = {
    "module__layers" : [1, 2],
    "module__activation" : [nn.ReLU(), nn.LeakyReLU()],
}

# Tune hyperparameters and train model
conv2lstm_estimator, conv2lstm_train_loss, conv2lstm_test_loss = get_model(conv2lstm, params, X, y, y_dict)

# Save hyperparameters for display
conv2lstm_params = conv2lstm_estimator.best_params_

# Save model
pickle.dump(conv2lstm_estimator, open("saved_models/conv2lstm_estimator.pkl", "wb"))

# Delete estimator to save space
del conv2lstm_estimator

Fitting 3 folds for each of 4 candidates, totalling 12 fits
  epoch    train_loss     dur
-------  ------------  ------
      1        0.0176  5.6725
      2        0.0174  5.5419
      3        0.0176  5.5419
      4        0.0174  5.5413
      5        0.0174  5.5411
      6        0.0174  5.5504
      7        0.0174  5.5454
      8        0.0174  5.5528
      9        0.0173  5.5526
     10        0.0173  5.5536
[CV] END ........module__activation=ReLU(), module__layers=1; total time=  56.3s
  epoch    train_loss     dur
-------  ------------  ------
      1        0.0182  5.5476
      2        0.0182  5.5550
      3        0.0182  5.5533
      4        0.0182  5.5583
      5        0.0182  5.5482
      6        0.0182  5.5523
      7        0.0182  5.5504
      8        0.0183  5.5557
      9        0.0183  5.5562
     10        0.0183  5.5557
[CV] END ........module__activation=ReLU(), module__layers=1; total time=  56.2s
  epoch    train_loss     dur
-------  ------------  -----

/home/jim/notebook_env/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:493: FitFailedWarning: 
6 fits failed out of a total of 12.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
6 fits failed with the following error:
Traceback (most recent call last):
  File "/home/jim/notebook_env/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 855, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jim/notebook_env/lib/python3.13/site-packages/skorch/net.py", line 1375, in fit
    self.partial_fit(X, y, _routing_method="fit", **fit_params)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jim/note

  epoch    train_loss     dur
-------  ------------  ------
      1        0.0177  8.3665
      2        0.0176  8.3178
      3        0.0176  8.3067
      4        0.0176  8.3071
      5        0.0176  8.3041
      6        0.0180  8.3099
      7        0.0182  8.3026
      8        0.0182  8.3086
      9        0.0182  8.3101
     10        0.0182  8.3039


In [14]:
print("==TRAIN LOSS==")
print(conv2lstm_train_loss)
print("==TEST LOSS==")
print(conv2lstm_test_loss)
print("==HYPERPARAMETERS==")
print(conv2lstm_params)

==TRAIN LOSS==
tensor(0.0183)
==TEST LOSS==
tensor(0.0183)
==HYPERPARAMETERS==
{'module__activation': ReLU(), 'module__layers': 1}
